# NB01_fPINN — Fractional (Caputo) damping in the 14-parameter delta plant: can α be identified?

**Purpose.** First step of the fractional-PINN paper. The baseline plant of notebook 18 (14-parameter structure + `τ_u` at severity `s`, payload scaling `γ(m_L)`) gets one extra term, a per-joint **Caputo damper** `τ_frac = B · D^α q`, with α ∈ (0, 1). Before training any network we check, with least squares only, that (1) the fractional term is visible in the data the networks of notebook 18 were trained on, and (2) α and B can be recovered.

**Avoiding the inverse crime.** The data generator and the identifier use different discretizations of `D^α`: generator = L1 scheme at the plant rate (0.5 ms) with full memory since t = 0; identifier = L1 on the logged, subsampled signal (2.5 ms, `SUBSAMPLE = 5` as in nb 18) with memory truncated to 1 s or full (`MEMORIES`).

**Models compared** (all fitted on the same nested 35 / 175 trajectories, same RNG as notebook 18):
`LS 14` · `LS 14 + friction` (per-joint viscous + Coulomb tanh, as NB03) · `LS 14 + friction + frac` (α by grid search, θ, friction and B by linear LS for each α).

**Outputs.** α̂ and B̂ against the true values for α ∈ {0.3, 0.5, 0.7, 0.9}; test torque RMSE (20 unseen random trajectories); on-policy torque RMSE along the Adept path at the three speeds (no closed loop yet).

In [ ]:
import time, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve
import matplotlib
import matplotlib.pyplot as plt

SEVERITIES = (0.0, 1.0)          # 0: plant = 14 params + fractional only; 1: + τ_u of nb 18
ALPHAS     = (0.3, 0.5, 0.7, 0.9)
B_FRAC     = 0.05                # [N m s^α] per joint, same order as b_v of τ_u
FS = 2000; dt = 1.0/FS           # plant rate, as in nb 18
SUBSAMPLE  = 5                   # logged data at 2.5 ms, as in nb 18
MEMORIES   = (1.0, None)         # identifier memory window [s]; None = full memory since t = 0
ALPHA_GRID = np.round(np.arange(0.05, 1.0001, 0.025), 3)
DATA_SEED, N_VAL_TRAJ, N_TEST_TRAJ, N_TRAIN_MAX = 12345, 20, 20, 175
SIZES = (35, 175)

## Robot model, unmodeled dynamics (identical to notebook 18)

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)

def gamma_payload(m): return (M_BASE + np.asarray(m, float)) / M_BASE

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def regressor_batch(q, a, P):
    """Vectorized regressor of NB01: (n,3) q, a; (n,3) P -> (n,3,14)."""
    n = len(q); s, c = np.sin(q), np.cos(q); x, y, z = P[:, 0], P[:, 1], P[:, 2]
    Y = np.zeros((n, 3, 14)); Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y

def tau_unmodeled_batch(q, qd, sev):
    u = UNMODELED
    return sev*(u["b_v"]*qd + u["f_c"]*np.tanh(qd/u["eps"])
                + u["c_cor"]*(qd**2*np.cos(q) + np.roll(qd, -1, axis=1)*np.roll(qd, -2, axis=1))
                + u["k_pos"]*np.sin(u["w_pos"]*q))

## Caputo derivative, L1 scheme

`D^α x(t_n) ≈ h^-α / Γ(2-α) · Σ_{k=0}^{n-1} b_k (x_{n-k} − x_{n-k-1})`, `b_k = (k+1)^{1-α} − k^{1-α}`, computed as one FFT convolution per signal. Optional memory window (short-memory principle). Checked on `x = t` (L1 is exact there: `D^α t = t^{1-α}/Γ(2-α)`) and on `x = t²`.

In [ ]:
def caputo_l1(x, h, alpha, window=None):
    """x: (N, d) samples on a uniform grid from t = 0 (system at rest before). Returns (N, d)."""
    x = np.asarray(x, float); N = len(x)
    K = N if window is None else min(N, int(window))
    k = np.arange(K); b = (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
    d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    out = fftconvolve(d, b[:, None], axes=0)[:N]
    return out * h**(-alpha) / Gamma(2 - alpha)

t = np.arange(4001)*dt
for al in (0.3, 0.7):
    e1 = np.abs(caputo_l1(t[:, None], dt, al)[:, 0] - t**(1-al)/Gamma(2-al)).max()
    e2 = np.abs(caputo_l1((t**2)[:, None], dt, al)[1:, 0] - 2*t[1:]**(2-al)/Gamma(3-al)).max()
    print(f"alpha {al}: max error on t = {e1:.1e}, on t^2 = {e2:.1e}")

## Data of notebook 18 (same RNG and trajectories) with the fractional term

`random_trajectory` is the one of notebook 18 but returns the full-rate signals, so the generator can apply `D^α` at 0.5 ms with full memory before the 2.5 ms logging and the margin cut. The identifier sees only the logged samples.

In [ ]:
def random_trajectory_full(rng, n_wp=6, v_range=(0.03, 0.8), t_min=0.3):     # RNG use identical to nb 18
    W = np.column_stack([rng.uniform(-0.12, 0.12, n_wp), rng.uniform(-0.12, 0.12, n_wp), rng.uniform(0.49, 0.56, n_wp)])
    dist = np.linalg.norm(np.diff(W, axis=0), axis=1)
    v = np.exp(rng.uniform(np.log(v_range[0]), np.log(v_range[1]), n_wp - 1))
    dur = np.maximum(dist / v, t_min); m_load = float(rng.integers(0, 7))
    edges = np.concatenate([[0.0], np.cumsum(dur)]); t_g = np.arange(0.0, edges[-1], dt)
    seg = np.clip(np.searchsorted(edges, t_g, side="right") - 1, 0, n_wp - 2)
    ts = (t_g - edges[seg]) / dur[seg]; s = 6*ts**5 - 15*ts**4 + 10*ts**3
    P = W[seg] + (W[seg + 1] - W[seg])*s[:, None]
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m_load)

def log_and_label(T, alpha, margin_s=0.05):
    """Generator: true torque at 0.5 ms with full-memory D^α; then logging at 2.5 ms and margin cut.
    Identifier-side D^α features for every alpha in ALPHA_GRID are computed from the LOGGED q only."""
    q, qd, qdd, P, m = T["q"], T["qd"], T["qdd"], T["P"], T["m"]
    g = gamma_payload(m)
    base = g*np.einsum("nij,j->ni", regressor_batch(q, qdd, P), THETA0) + B_FRAC*caputo_l1(q, dt, alpha)
    tu = tau_unmodeled_batch(q, qd, 1.0)
    mg = int(margin_s/dt); idx = np.arange(0, len(q), SUBSAMPLE)       # logging grid from t = 0
    keep = (idx >= mg) & (idx < len(q) - mg)
    ql = q[idx]; h = dt*SUBSAMPLE
    i = idx[keep]
    out = dict(q=q[i], qd=qd[i], qdd=qdd[i], P=P[i], m=np.full(len(i), m), base=base[i], tu=tu[i])
    for mem in MEMORIES:                                                   # (n_alpha, n, 3) per memory
        out[f"D_{mem}"] = np.stack([caputo_l1(ql, h, a, window=None if mem is None else mem/h)[keep] for a in ALPHA_GRID])
    return out

def build_sets(alpha):
    rng = np.random.default_rng(DATA_SEED)
    tr = [log_and_label(random_trajectory_full(rng), alpha) for _ in range(N_TRAIN_MAX)]
    _ = [random_trajectory_full(rng) for _ in range(N_VAL_TRAJ)]          # validation set consumed as in nb 18
    te = [log_and_label(random_trajectory_full(rng), alpha) for _ in range(N_TEST_TRAJ)]
    return tr, te

def cat(parts):
    return {k: np.concatenate([p[k] for p in parts], axis=1 if k.startswith("D_") else 0) for k in parts[0]}

def view(S, sev, mem):
    """Dataset as the identifier sees it: label for severity sev, D^α features for memory mem."""
    return dict(q=S["q"], qd=S["qd"], qdd=S["qdd"], P=S["P"], m=S["m"], y=S["base"] + sev*S["tu"], D=S[f"D_{mem}"])

## Least-squares models

Rows = samples × joints. Columns: `γ Y` (14, shared by the three joints), per-joint viscous `q̇_i` and Coulomb `tanh(q̇_i/0.05)` (6), per-joint fractional `D^α q_i` (3). For the fractional model α is chosen by grid search on the training residual (variable projection); B̂ is the mean of the three per-joint coefficients.

In [ ]:
def base_features(D, kind):
    """All columns except the fractional ones."""
    g = gamma_payload(D["m"])[:, None, None]
    cols = [g*regressor_batch(D["q"], D["qdd"], D["P"])]
    n = len(D["q"]); E = np.eye(3)[None]
    if kind in ("fric", "frac"):
        cols += [D["qd"][:, :, None]*E, np.tanh(D["qd"]/0.05)[:, :, None]*E]
    if kind in ("exact", "exact_frac"):                                   # exact functional forms of τ_u (NB03 bound)
        q, qd = D["q"], D["qd"]
        cols += [np.stack([qd, np.tanh(qd/UNMODELED["eps"]),
                           qd**2*np.cos(q) + np.roll(qd, -1, axis=1)*np.roll(qd, -2, axis=1),
                           np.sin(UNMODELED["w_pos"]*q)], axis=2)]
    return np.concatenate(cols, axis=2).reshape(3*n, -1)

def frac_columns(D, ia):
    return (D["D"][ia][:, :, None]*np.eye(3)[None]).reshape(-1, 3)

def features(D, kind, ia=None):
    A = base_features(D, kind)
    return np.hstack([A, frac_columns(D, ia)]) if kind in ("frac", "exact_frac") else A

def _solve(G, g):
    s = 1/np.sqrt(np.maximum(np.diag(G), 1e-300))                        # column scaling (θ spans 1e-5 ... 1)
    return s*np.linalg.lstsq(G*s[:, None]*s[None, :], g*s, rcond=1e-12)[0]

def fit(D, kind):
    """Linear LS through the Gram matrix; for fractional models α by grid search (variable projection).
    The fixed block A0'A0 is formed once and only the 3 fractional columns change with α."""
    y = D["y"].reshape(-1); A0 = base_features(D, kind); G00, g0, yy, n = A0.T @ A0, A0.T @ y, y @ y, len(y)
    if kind not in ("frac", "exact_frac"):
        return dict(kind=kind, w=_solve(G00, g0), ia=None)
    best = None; prof = []
    for ia in range(len(ALPHA_GRID)):
        A1 = frac_columns(D, ia)
        G = np.block([[G00, A0.T @ A1], [A1.T @ A0, A1.T @ A1]]); g = np.concatenate([g0, A1.T @ y])
        w = _solve(G, g); r = np.sqrt(max(yy - 2*w @ g + w @ G @ w, 0.0)/n); prof.append(r)
        if best is None or r < best[0]: best = (r, ia, w)
    return dict(kind=kind, w=best[2], ia=best[1], profile=np.array(prof))

def rmse(model, D):
    A = features(D, model["kind"], model["ia"])
    return float(np.sqrt(np.mean((A @ model["w"] - D["y"].reshape(-1))**2)))

## Adept references (NB01) for the on-policy torque check

The fractional features along the reference are computed by the identifier rule (2.5 ms, same memory window) for every grid α, and the true torque by the generator rule.

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05
SPEED_SCALES = (6.4, 3.2, 1.6)

def adept_full(scale, m=0.0):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]
    P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m)

## Run: severity × memory window × α × data size

Models: `LS 14`, `LS 14 + friction`, `LS 14 + friction + frac`, and the NB03 upper bound `LS exact τ_u forms` with and without the fractional columns (a practitioner would not know those forms; they show what is identifiable once the rest of the residual is captured, which is the job of the PINN).

In [ ]:
MODELS = (("ls14", "LS 14"), ("fric", "LS 14 + friction"), ("frac", "LS 14 + friction + frac"),
          ("exact", "LS exact forms (bound)"), ("exact_frac", "LS exact forms + frac (bound)"))
rows = []; t_all = time.time(); profiles = {}
for alpha in ALPHAS:
    t0 = time.time(); tr, te = build_sets(alpha); TE = cat(te)
    refs = {sc: log_and_label(adept_full(sc), alpha, margin_s=0.0) for sc in SPEED_SCALES}
    for size in SIZES:
        TRS = cat(tr[:size])
        for sev in SEVERITIES:
            for mem in MEMORIES:
                TR = view(TRS, sev, mem)
                for kind, label in MODELS:
                    if kind in ("ls14", "fric", "exact") and mem != MEMORIES[0]: continue   # no D^α columns: one memory is enough
                    M = fit(TR, kind)
                    row = {"Severity": sev, "Memory (s)": "full" if mem is None else mem, "alpha true": alpha,
                           "Data (traj.)": size, "Model": label, "Train RMSE (N m)": rmse(M, TR),
                           "Test RMSE (N m)": rmse(M, view(TE, sev, mem))}
                    for sc in SPEED_SCALES: row[f"Adept {sc} RMSE (N m)"] = rmse(M, view(refs[sc], sev, mem))
                    if "profile" in M:
                        row["alpha hat"] = ALPHA_GRID[M["ia"]]; row["B hat (N m s^a)"] = M["w"][-3:].mean()
                        row["B hat spread"] = np.ptp(M["w"][-3:])
                        profiles[(kind, sev, mem, alpha, size)] = M["profile"]
                    rows.append(row)
    print(f"alpha {alpha}: {len(TRS['q'])} logged samples in {max(SIZES)} traj | {time.time()-t0:.0f} s", flush=True)
    del tr, te, TE, TRS
df = pd.DataFrame(rows); df.to_csv("NB01_fPINN_FracLS_AllRuns.csv", index=False)
print(f"total {(time.time()-t_all)/60:.1f} min")

## Summary: is α identified?

In [ ]:
pd.set_option("display.width", 250)
fr = df[df.Model.str.contains("frac")]
pa = fr.pivot_table(index=["Model", "Severity", "Memory (s)", "Data (traj.)"], columns="alpha true", values="alpha hat")
pb = fr.pivot_table(index=["Model", "Severity", "Memory (s)", "Data (traj.)"], columns="alpha true", values="B hat (N m s^a)")
print("alpha hat (columns = true alpha):"); print(pa.to_string())
print(f"\nB hat, mean of the 3 joints (true B = {B_FRAC}):"); print(pb.round(4).to_string())

## Summary: torque error

In [ ]:
sub = df[df["Data (traj.)"] == max(SIZES)]
for col in ["Test RMSE (N m)", "Adept 1.6 RMSE (N m)"]:
    pv = sub.pivot_table(index=["Severity", "Model", "Memory (s)"], columns="alpha true", values=col)
    print(f"\n{col.replace('(N m)', '(mN m)')}, {max(SIZES)} trajectories:"); print((1e3*pv).round(2).to_string())

## Figure: training residual vs grid α (175 trajectories, full memory)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))
for ax, (kind, sev, ttl) in zip(axes, (("frac", 0.0, "s = 0, LS 14 + friction + frac"), ("exact_frac", 1.0, "s = 1, exact forms + frac"))):
    for alpha in ALPHAS:
        pr = profiles[(kind, sev, None, alpha, max(SIZES))]
        ax.plot(ALPHA_GRID, pr/pr.min(), lw=1, label=f"true α = {alpha}")
        ax.axvline(alpha, lw=0.6, ls=":", color=ax.lines[-1].get_color())
    ax.set_yscale("log"); ax.set_xlabel("Grid α"); ax.set_title(ttl, fontsize=8); ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel("Train RMSE / min"); axes[0].legend(fontsize=6)
fig.tight_layout(); fig.savefig("fig_nb01_fpinn_alpha_profile.pdf"); fig.savefig("fig_nb01_fpinn_alpha_profile.png", dpi=200)
plt.show()